# Gerador de Cartas de Magic: The Gathering

Projeto da disciplina de IA e Bibliotecas de Dados para Python.

**Como usar:** no menu acima, clique em **Ambiente de execução > Executar tudo** (ou `Ctrl+F9`). As duas primeiras células preparam o ambiente (clonam o repositório e instalam as dependências); a terceira mostra o formulário para gerar a carta.

O notebook gera a mesma carta de **três formas** — os três braços do estudo comparativo:

| Braço | Como o texto é gerado | Precisa de Ollama? |
|---|---|---|
| **M1** | cadeia de Markov + regressões (custo e P/T) | não |
| **M2** | o mesmo M1, com o **texto de regras** refinado por LLM | sim |
| **M3** | LLM pura (`gemma3:4b`), com schema restrito aos seis tipos | sim |

A comparação entre eles está em [`experimentos/relatorio_metricas.md`](../experimentos/relatorio_metricas.md) (estrutura e texto, 1.000 cartas por braço) e [`experimentos/relatorio_modo_b.md`](../experimentos/relatorio_modo_b.md) (inferência de tipo pelo nome).

O código completo do pipeline de geração está em [`src/pipeline/generator.py`](../src/pipeline/generator.py) e o dos braços M2/M3 em [`src/pipeline/llm.py`](../src/pipeline/llm.py). Este notebook só importa e chama as funções, para que a mesma lógica possa também ser usada por outras interfaces no futuro (ex: um app web).


## 1. Preparar o ambiente

Clona o repositório (se ainda não estiver presente) e instala as dependências. O `requirements.txt` fixa a versão do **scikit-learn** (`1.9.1`) usada para treinar os modelos de custo/P-T, para que o `models/modelos_estruturais.pkl` carregue sem avisos de versão incompatível. Se você já tinha o Colab aberto antes desta mudança, use **Ambiente de execução > Reiniciar sessão** e rode tudo de novo, para o ambiente pegar a versão certa.

In [ ]:
import os

REPO_URL = "https://github.com/funkymonks21/MTGCardCreator-IFRS.git"
REPO_DIR = "MTGCardCreator-IFRS"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}
else:
    # a pasta ja existe de uma execucao anterior nesta sessao do Colab --
    # puxa os commits novos em vez de ignorar
    !cd {REPO_DIR} && git pull -q

%cd {REPO_DIR}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "src")

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt_tab", quiet=True)
# POS tagging usado pela inferencia automatica de tipo (ver
# src/pipeline/tipo_por_nome.py). Os dois nomes cobrem versoes
# diferentes do NLTK.
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

print("Ambiente pronto.")


## 2. Carregar o pipeline

Importa as funções dos três braços. Dois conjuntos de modelos são carregados automaticamente, sob demanda, direto do repositório, então ninguém precisa treinar nada aqui:

- **texto** -- cadeia de Markov treinada por tipo e cor (`models/modelo_markov.pkl`, ver `src/pipeline/texto_markov.py`);
- **custo de mana e poder/resistência** -- regressões scikit-learn (`models/modelos_estruturais.pkl`, ver `src/pipeline/estrutura.py`).

O pipeline decide primeiro o **tipo** (Criatura, Instantânea, Feitiço, Artefato, Encantamento ou Terreno), depois gera as habilidades com o modelo daquele tipo e só então prevê custo e poder/resistência. Instants/sorceries não têm habilidades ativadas nem P/T; terrenos não têm custo de mana nem P/T.

Com a opção **Automático (pelo nome)**, o tipo é inferido a partir do nome usando POS tagging + WordNet (ver `src/pipeline/tipo_por_nome.py`): verbos/ações puxam instantânea/feitiço, lugares puxam terreno, profissões/nomes próprios puxam criatura, objetos puxam artefato e conceitos abstratos puxam encantamento. Se houver empate ou nenhuma pista, o tipo é sorteado.

### Os três modelos

- **M1 — pipeline.** É o que você já vê nas seções seguintes: Markov por tipo e cor, mais as regressões de custo e P/T. Roda offline.
- **M2 — pipeline + refino.** Gera exatamente a **mesma estrutura** do M1 (mesma semente ⇒ mesmo tipo, subtipos, custo e P/T) e manda só o `texto_regras` para o `gemma3:4b` refinar. É o braço que isola o efeito do texto.
- **M3 — LLM pura.** O `gemma3:4b` gera a carta inteira (nome, custo, texto, P/T, keywords) em JSON, com o tipo restrito aos seis do escopo. O tipo é resolvido pelo pipeline e **imposto** ao modelo, para que os três braços produzam cartas comparáveis — o mesmo desenho do Modo A do estudo.

M2 e M3 usam o **Ollama** local (`localhost:11434`). No Colab não há Ollama: o M1 funciona, e os outros dois mostram uma mensagem explicando o que falta. Para rodar tudo na sua máquina:

```bash
ollama serve
ollama pull gemma3:4b
```


In [ ]:
from pipeline import (
    gerar_carta,
    gerar_com_modelo,
    gerar_carta_m2,
    gerar_carta_m3,
    ollama_disponivel,
    MODELOS,
    ROTULO_MODELO,
    FalhaGeracao,
    OllamaIndisponivel,
    COLOR_PIE,
    PESO_COR_PADRAO,
    PESOS_NGRAMA_PADRAO,
    TIPO_ALEATORIO,
    TIPO_AUTOMATICO,
    TIPOS_DISPONIVEIS,
)

print("Pipeline carregado. Cores disponiveis:", list(COLOR_PIE))
print("Tipos disponiveis:", list(TIPOS_DISPONIVEIS))
print("Modelos disponiveis:", list(MODELOS))
print("Peso padrao da cor na geracao (0=caos total, 1=so vocabulario da cor):", PESO_COR_PADRAO)
print("Pesos padrao por ordem de n-grama (0=unigrama ... 4=pentagrama):", PESOS_NGRAMA_PADRAO)

if ollama_disponivel():
    print("Ollama: disponivel -- M2 e M3 prontos.")
else:
    print("Ollama: INDISPONIVEL -- so o M1 vai funcionar.")
    print("  Para liberar M2/M3: `ollama serve` e `ollama pull gemma3:4b`.")


### Teste rápido do M1 (sem usar os campos abaixo)

O M1 é a base dos três braços e o único que roda sem Ollama. A geração é uma cadeia de Markov (unigramas até pentagramas) treinada por **tipo e cor**, com peso ajustável entre o vocabulário típico da cor e um modelo geral -- **sem** filtro de color pie, de propósito (ver decisão de design em `src/pipeline/generator.py`). Rodar a mesma carta várias vezes deve dar textos diferentes.

O tipo pode ser escolhido, inferido pelo nome (`Automático (pelo nome)`) ou sorteado (`Aleatório`). No modo automático, nomes como "Serra Angel" (profissão), "Forest" (lugar), "Sword" (objeto) e "Destroy" (verbo) já sugerem o tipo; nomes sem pista caem no sorteio.

Os exemplos abaixo mostram também o `tipo_origem`: ele diz se o tipo foi **escolhido**, **inferido pelo nome** ou **sorteado** — a mesma decisão que o M3 recebe pronta quando é chamado pela interface.


In [ ]:
exemplos = [
    ("Murder", "Preto", 0.7, TIPO_AUTOMATICO),             # verbo/ação -> instantânea/feitiço
    ("Serra Angel", "Branco", 0.7, TIPO_AUTOMATICO),       # profissão -> criatura
    ("Floresta Ancestral", "Verde", 0.7, TIPO_AUTOMATICO), # lugar -> terreno
    ("Mind Stone", "Azul", 0.7, TIPO_AUTOMATICO),          # objeto -> artefato
    ("Pacifism", "Branco", 0.7, TIPO_AUTOMATICO),          # conceito abstrato -> encantamento
    ("Lightning Bolt", "Vermelho", 0.7, TIPO_AUTOMATICO),  # fenômeno -> instantânea
    ("jasdhahs12123", "Verde", 0.7, TIPO_ALEATORIO),       # sem pista -> tipo sorteado
]

for nome_exemplo, cor_exemplo, peso_exemplo, tipo_exemplo in exemplos:
    c = gerar_carta(
        nome_exemplo, cor_exemplo, peso_cor=peso_exemplo, tipo=tipo_exemplo
    )
    print(f"{nome_exemplo!r} + {cor_exemplo} + {tipo_exemplo} (peso_cor={peso_exemplo}):")
    print(f"  tipo: {c.tipo_linha} | custo: {c.custo_mana or '(nenhum -- terreno)'}")
    print(f"  como o tipo foi decidido: {c.tipo_origem} -- {c.tipo_motivo}")
    if c.poder is not None:
        print(f"  P/T: {c.poder}/{c.resistencia}")
    print(f"  semente: {c.semente_usada or '(nenhuma)'}")
    print(f"  texto: {c.texto_regras}\n")


## 3. Gerar sua carta

Preencha o nome, escolha o **modelo** (M1, M2 ou M3), a cor e o tipo, ajuste o quanto a cor deve pesar na geração, depois clique em **Gerar carta**.

Os controles aparecem **à esquerda** e a **carta gerada à direita**, lado a lado — assim dá para mexer nos sliders e ver o resultado da carta ao mesmo tempo.

- **M1 — pipeline**: todos os controles valem. Texto vem do Markov; custo e P/T das regressões.
- **M2 — pipeline + refino**: os mesmos controles valem (a estrutura é do pipeline). A diferença é que o texto de regras passa pelo LLM antes de aparecer. Se o refino falhar, a carta mantém o texto do M1 e o painel avisa.
- **M3 — LLM pura**: peso da cor, parágrafos e n-gramas ficam **desligados**, porque o texto vem inteiro do modelo. Nome, cor e tipo continuam valendo: o tipo é resolvido pelo pipeline e imposto ao LLM.

A **semente** controla a reprodutibilidade. Com `-1`, cada clique sorteia; com um número fixo, a mesma entrada dá a mesma carta — e, em M1 e M2, exatamente a **mesma estrutura** (o que torna a diferença de texto visível).

- **Automático (pelo nome)** (padrão): tenta inferir o tipo pelas palavras do nome; se não houver pista ou houver empate, sorteia.
- **Aleatório**: sorteia direto, sem inferência (criaturas são mais comuns, como no corpus real).
- Os demais itens forçam um tipo específico.

Cada tipo recebe subtipos sorteados a partir das probabilidades do corpus (ex.: **Creature — Human Warrior**, **Artifact — Equipment**, **Land — Forest Island**). Criaturas têm 1–3 subtipos, artefatos 0–1, terrenos 0–2, encantamentos 0–2 e mágicas 0–1.

Auras sempre têm uma linha **Enchant ...**, Equipamentos sempre têm **Equipped creature ...** e **Equip ...** (o efeito concedido e o custo de equipar), **Veículos** sempre têm **Crew N** e poder/resistência, **Sagas** têm cada parágrafo como um capítulo numerado em romano (**I — ...**, **II — ...**), e terrenos com tipo básico ganham a habilidade de mana correspondente (ex.: **{T}: Add {G}** para Forest).

Terrenos são gerados sem custo de mana e sem P/T; instants e feitiços não têm habilidades ativadas nem P/T.

Cada carta tem **vários parágrafos de habilidade**: a quantidade é sorteada da **distribuição real** do tipo no corpus (criaturas costumam ter 1–2, terrenos 2–3 etc.). Em Sagas, esse número é o de **capítulos**. O controle **Parágrafos** deixa fixar um número (ou deixar `Automático`). Os **keywords** também são sorteados a partir dos dados, com um fator que deixa a carta com mais keywords que a média real (que é baixa).

O texto do M1 é gerado por uma **cadeia de Markov** que combina várias ordens de n-grama: **unigrama, bigrama, trigrama, tetragrama e pentagrama**. Cada ordem tem um slider de peso (0 a 1): quanto maior, mais aquela ordem "puxa" na escolha da próxima palavra. Contextos mais longos (tetra/penta) só existem quando aquela sequência realmente apareceu no corpus, então são o sinal mais forte de coerência — por isso vêm com peso maior por padrão.

### Arte e moldura

Marque **Gerar arte + moldura realista (API de imagem)** para que a carta sai com **ilustração** e **moldura** de verdade. Nesse modo o pipeline monta um prompt a partir do **nome**, **tipo** e **texto** da carta, baixa a arte de uma API de imagem gratuita (Pollinations.ai, sem chave) e desenha a moldura em código — painel do título com gradiente da cor, caixa de tipo, caixa de texto translúcida e selo de poder/resistência — usando fontes livres (Cinzel e EB Garamond).

A primeira geração de um nome novo leva alguns segundos (a API gera a imagem na hora); artes repetidas saem do cache local. Se não houver internet, a carta ainda é gerada, com um fundo procedural no lugar da arte — a geração **nunca** quebra por causa da API.

Sem o checkbox (padrão), a figura sai num placeholder simples, instantâneo e offline — bom para gerar várias cartas em sequência ou testar o texto.


In [ ]:
import ipywidgets as widgets
from IPython.display import Image as ImagemIPython
from IPython.display import clear_output, display

# --- Seletor de modelo: os tres bracos do estudo -----------------------
campo_modelo = widgets.Dropdown(
    options=[(ROTULO_MODELO[m], m) for m in MODELOS],
    value="M1",
    description="Modelo:",
    style={"description_width": "initial"},
    tooltip=(
        "M1 = pipeline (Markov + regressao). M2 = o mesmo M1 com o texto "
        "refinado por LLM. M3 = LLM pura. M2 e M3 exigem o Ollama no ar."
    ),
)
aviso_ollama = widgets.HTML()
dica_modelo = widgets.HTML()


def _atualizar_aviso_ollama():
    """Mostra, uma vez, se o Ollama responde -- sem travar a interface."""
    if ollama_disponivel():
        aviso_ollama.value = (
            "<span style='color:#1a7f37'>Ollama disponivel</span> "
            "-- M2 e M3 prontos."
        )
    else:
        aviso_ollama.value = (
            "<span style='color:#b35900'>Ollama indisponivel</span> "
            "-- so o M1 funciona. Para M2/M3: <code>ollama serve</code> e "
            "<code>ollama pull gemma3:4b</code>."
        )


_atualizar_aviso_ollama()

campo_nome = widgets.Text(
    value="Serra Angel",
    description="Nome:",
    placeholder="ex: Murder, Serra Angel, Forest, Sword...",
    style={"description_width": "initial"},
)
campo_cor = widgets.Dropdown(
    options=list(COLOR_PIE.keys()),
    description="Cor:",
    style={"description_width": "initial"},
)
campo_tipo = widgets.Dropdown(
    options=list(TIPOS_DISPONIVEIS),
    value=TIPO_AUTOMATICO,
    description="Tipo:",
    style={"description_width": "initial"},
)
campo_peso_cor = widgets.FloatSlider(
    value=PESO_COR_PADRAO, min=0.0, max=1.0, step=0.1,
    description="Peso da cor:",
    style={"description_width": "initial"},
    readout_format=".1f",
)

# Quantos PARAGRAFOS de habilidade a carta tera. Por padrao e sorteado da
# distribuicao real do tipo (criatura costuma ter 1-2, terreno 2-3 etc.).
# 'Automatico' deixa o sorteio acontecer; escolher um numero forca aquele
# total, util para demonstrar o efeito na apresentacao.
campo_n_paragrafos = widgets.Dropdown(
    options=[("Automatico (pelos dados)", None)] + [(str(n), n) for n in range(1, 6)],
    value=None,
    description="Paragrafos:",
    style={"description_width": "initial"},
    tooltip="Numero de paragrafos de habilidade. Automatico sorteia da distribuicao real do tipo.",
)

# Semente: -1 sorteia; um numero fixo torna a carta reproduzivel. Em M1/M2 a
# MESMA semente da a mesma estrutura -- o que deixa a diferenca de texto
# visivel na apresentacao.
campo_seed = widgets.IntText(
    value=-1,
    description="Semente:",
    style={"description_width": "initial"},
    tooltip=(
        "Use -1 para sortear. Com um numero fixo, a mesma entrada gera a mesma "
        "carta (M1 e M2 compartilham a estrutura, porque usam a mesma semente)."
    ),
)

# Um slider por ORDEM de n-grama. A chave e o tamanho do contexto que a
# ordem olha para tras: 0=unigrama (sem contexto), 1=bigrama, 2=trigrama,
# 3=tetragrama, 4=pentagrama. Os pesos sao RELATIVOS: cada passo de geracao
# so mistura as ordens que TEM dados para o contexto e cujo peso seja > 0.
_rotulos_ngrama = {
    0: "Unigrama (0)",
    1: "Bigrama (1)",
    2: "Trigrama (2)",
    3: "Tetragrama (3)",
    4: "Pentagrama (4)",
}
campo_pesos_ngrama = {}
for _ordem, _rotulo in _rotulos_ngrama.items():
    campo_pesos_ngrama[_ordem] = widgets.FloatSlider(
        value=PESOS_NGRAMA_PADRAO.get(_ordem, 0.0), min=0.0, max=1.0, step=0.05,
        description=_rotulo,
        style={"description_width": "initial"},
        readout_format=".2f",
        tooltip="Peso desta ordem de n-grama na escolha da proxima palavra (pesos sao relativos entre as ordens disponiveis)",
    )
# Gerar a ILUSTRACAO pela API de imagem e desenhar a moldura realista. Por
# padrao fica desligado (gera em ~instantaneo, offline, com um placeholder);
# ligado, o pipeline monta um prompt a partir do nome/tipo/texto e baixa a
# arte (Pollinations.ai, sem chave). A primeira geracao de um nome novo leva
# alguns segundos; artes ja baixadas ficam em cache (assets/cache_arte).
campo_usar_arte = widgets.Checkbox(
    value=False,
    description="Gerar arte + moldura realista (API de imagem)",
    style={"description_width": "initial"},
    tooltip=(
        "Liga a arte via API (Pollinations.ai) e a composicao da moldura. "
        "O plano gratuito permite ~1 imagem a cada 45s; se o limite for "
        "atingido (ou faltar internet), a carta usa um fundo procedural "
        "local e avisa o motivo."
    ),
)
botao_gerar = widgets.Button(description="Gerar carta", button_style="success")
saida = widgets.Output()


def _pesos_ngrama_dos_campos():
    """Le os sliders e devolve o dict {ordem: peso} atual."""
    return {ordem: campo.value for ordem, campo in campo_pesos_ngrama.items()}


def ao_trocar_modelo(_mudanca=None):
    """Desliga os controles que nao afetam o M3 (o texto vem do LLM)."""
    m3 = campo_modelo.value == "M3"
    for campo in [campo_peso_cor, campo_n_paragrafos, *campo_pesos_ngrama.values()]:
        campo.disabled = m3
    if m3:
        dica_modelo.value = (
            "<i style='color:#6b6459'>M3: peso da cor, paragrafos e n-gramas "
            "nao se aplicam -- o texto vem inteiro do LLM. Nome, cor e tipo "
            "(resolvido pelo pipeline) sao impostos ao modelo.</i>"
        )
    elif campo_modelo.value == "M2":
        dica_modelo.value = (
            "<i style='color:#6b6459'>M2: a estrutura e a do pipeline (mesma "
            "semente = mesma carta do M1); so o texto de regras passa pelo "
            "refino do LLM.</i>"
        )
    else:
        dica_modelo.value = (
            "<i style='color:#6b6459'>M1: texto por cadeia de Markov; custo e "
            "P/T pelas regressoes. Roda offline.</i>"
        )


def ao_clicar(_botao):
    with saida:
        clear_output(wait=True)
        modelo = campo_modelo.value
        seed = None if campo_seed.value < 0 else int(campo_seed.value)
        # Guarda as mensagens da arte (ex.: limite gratuito da API atingido)
        # para mostrar logo abaixo da carta, em vez de cair no fundo liso
        # sem nenhuma explicacao.
        avisos_arte = []
        try:
            carta = gerar_com_modelo(
                modelo,
                campo_nome.value,
                campo_cor.value,
                peso_cor=campo_peso_cor.value,
                tipo=campo_tipo.value,
                pesos_ngrama=_pesos_ngrama_dos_campos(),
                n_paragrafos=campo_n_paragrafos.value,
                usar_arte=campo_usar_arte.value,
                avisar_arte=avisos_arte.append,
                seed=seed,
            )
        except OllamaIndisponivel as exc:
            print("O M2/M3 precisa do Ollama, e a chamada nao funcionou.")
            print()
            print(str(exc))
            print()
            print("Sem Ollama, use o M1: a geracao continua funcionando offline.")
            return
        except FalhaGeracao as exc:
            print("O M3 nao produziu uma carta valida nesta tentativa.")
            print(str(exc))
            print()
            print("A amostragem do LLM e estocastica: clique de novo para tentar outra.")
            return

        # Exibimos via IPython.Image com largura fixa: garante que a carta
        # inteira apareca (com a moldura) sem ser cortada nem esticada na
        # largura da coluna do notebook.
        import io as _io

        _buf = _io.BytesIO()
        carta.imagem_final.save(_buf, format="PNG")
        display(ImagemIPython(data=_buf.getvalue(), width=360))
        if avisos_arte:
            print("AVISO: " + " ".join(avisos_arte))
        print(f"Modelo: {ROTULO_MODELO[carta.modelo]}")
        print(f"Nome: {carta.nome}")
        print(f"Tipo: {carta.tipo_linha}")
        print(f"Como o tipo foi decidido: {carta.tipo_origem} -- {carta.tipo_motivo}")
        print(f"Custo de mana: {carta.custo_mana or '(nenhum -- terreno)'}")
        if carta.poder is not None:
            print(f"Poder/Resistencia: {carta.poder}/{carta.resistencia}")
        print(f"Texto: {carta.texto_regras}")
        print()
        if carta.modelo == "M2":
            estado = "sim" if carta.refinado else "nao (texto do pipeline mantido)"
            print(f"[M2] refinamento aplicado: {estado}")
        if carta.modelo in ("M1", "M2"):
            print(f"[debug] paragrafos de habilidade: {carta.n_paragrafos}")
            print(f"[debug] keywords: {carta.palavras_chave or '(nenhuma)'}")
            print(f"[debug] peso da cor usado: {carta.peso_cor}")
            print(f"[debug] pesos por ordem de n-grama usados: {carta.pesos_ngrama}")
            print(f"[debug] palavra-semente: {carta.semente_usada or '(nenhuma -- cadeia comecou do zero)'}")
        else:
            print(f"[M3] keywords: {carta.palavras_chave or '(nenhuma)'}")
            print(f"[M3] semente usada: {carta.semente_usada}")
        if campo_usar_arte.value:
            de_api = "API" if carta.arte_da_api else "fallback local"
            detalhe = f" (motivo: {carta.arte_motivo})" if carta.arte_motivo else ""
            print(f"[debug] arte: {de_api}{detalhe}")


campo_modelo.observe(ao_trocar_modelo, names="value")
ao_trocar_modelo()
botao_gerar.on_click(ao_clicar)

# --- Layout em duas colunas: controles a esquerda, carta a direita ------
# Assim os sliders ficam visiveis ao mesmo tempo que a carta gerada, o que
# ajuda na apresentacao. Cada coluna usa layout flexivel do ipywidgets.
titulo_controles = widgets.HTML(
    "<b style='font-size:15px'>Controles</b>"
)
coluna_controles = widgets.VBox(
    [
        titulo_controles,
        aviso_ollama,
        campo_modelo,
        dica_modelo,
        campo_nome,
        campo_cor,
        campo_tipo,
        campo_seed,
        campo_peso_cor,
        campo_n_paragrafos,
        campo_usar_arte,
        widgets.HTML("<b>Pesos dos n-gramas</b>"),
        *[campo_pesos_ngrama[o] for o in sorted(campo_pesos_ngrama)],
        botao_gerar,
    ],
    layout=widgets.Layout(
        width="48%", padding="12px", border="1px solid #d0d0d0",
        margin="0 1% 0 0", align_items="stretch",
    ),
)

titulo_carta = widgets.HTML(
    "<b style='font-size:15px'>Carta gerada</b>"
)
coluna_carta = widgets.VBox(
    [titulo_carta, saida],
    layout=widgets.Layout(
        width="48%", padding="12px", border="1px solid #d0d0d0",
        align_items="stretch",
    ),
)

layout_duas_colunas = widgets.HBox(
    [coluna_controles, coluna_carta],
    layout=widgets.Layout(width="100%", align_items="flex-start"),
)
display(layout_duas_colunas)

# Gera uma carta de exemplo ja no carregamento, para a coluna da direita nao
# ficar vazia. M1 por padrao -- nao depende do Ollama.
ao_clicar(None)


### Calibrar os pesos dos n-gramas

Cada slider de peso controla uma **ordem de n-grama** na escolha da próxima palavra (os pesos são relativos: só entram as ordens que têm dados para o contexto atual):

- **Unigrama (0)**: frequência geral das palavras — o "chão" do modelo, não olha contexto nenhum;
- **Bigrama (1)**: olha a última palavra;
- **Trigrama (2)**: olha as duas últimas palavras;
- **Tetragrama (3)**: olha as três últimas palavras;
- **Pentagrama (4)**: olha as quatro últimas palavras.

Ordens mais longas (tetra/penta) só fazem algo quando aquela sequência **existe** no corpus — e aí são o sinal mais forte de coerência. Ordens curtas dão mais variedade. Baixar um peso para **0** remove aquela ordem da mistura.

A célula abaixo gera o **mesmo texto com vários esquemas de peso**, usando uma semente fixa (mesma palavra-semente), para você comparar lado a lado. Ajuste `COR`/`TIPO`/`SEMENTE` se quiser testar outros contextos.


In [ ]:
from pipeline.texto_markov import carregar_modelo

COR = "Vermelho"       # cor para calibrar
TIPO = "Criatura"      # "Criatura", "Instantânea", "Feitiço", "Artefato", "Encantamento", "Terreno"
SEMENTE = 42           # mantém a mesma cadeia; troque para ver outros textos

from pipeline import COLOR_PIE, TIPOS_DISPONIVEIS  # noqa: F811

cor_abrev = COLOR_PIE[COR]["abrev"]
tipo_chave = TIPOS_DISPONIVEIS[TIPO]
_modelo = carregar_modelo()

# Cada esquema mapeia ordem -> peso. Lembre: só as ordens com dados para o
# contexto e peso > 0 entram na mistura de cada passo.
esquemas = [
    ("padrão (favorece contexto longo)", PESOS_NGRAMA_PADRAO),
    ("só bigrama", {1: 1.0}),
    ("só trigrama", {2: 1.0}),
    ("só tetragrama + pentagrama", {4: 1.0, 3: 1.0}),
    ("todos com peso igual", {4: 1.0, 3: 1.0, 2: 1.0, 1: 1.0, 0: 1.0}),
]

for rotulo, pesos in esquemas:
    r = _modelo.gerar_texto(
        cor_abrev, tipo=tipo_chave, peso_cor=0.7, seed=SEMENTE, pesos_ngrama=pesos,
    )
    print(f"--- {rotulo} ---")
    print(f"    pesos: {pesos}")
    print(r["texto"])
    print()
